# iPSC-EC day 60 — endothelial vs fibroblast marker violins

Per-gene violin plots of six endothelial and six fibroblast markers across the two
day-60 iPSC-EC media conditions (5% FBS gfp vs. VEGF+SB+3% FBS), three biological
replicates each. Violins pool the replicates per condition; individual replicates are
overlaid as dots and the inner box shows the median and quartiles.

**Pipeline** (matching the Figure 1E marker-panel notebook):

1. Load the Cuffdiff FPKM matrix and index it by `gene_official`, keeping the first
   entry for duplicate gene symbols.
2. Select the two day-60 conditions and the curated marker panel.
3. Transform with log10(FPKM + 1).

**Statistics**: significance is taken from the Cuffdiff v2.2.1 run for this comparison
(`sample1.vs.sample3`), which models fragment counts with a beta negative binomial
distribution across the three biological replicates per condition and tests the log fold
change in FPKM against its standard error. Nominal (uncorrected) p-values are shown for
this pre-specified marker panel. Set `SHOW_STATS = False` to omit the brackets.

The replicate dots are jittered with the global NumPy RNG; `RNG` is fixed so repeat runs
are byte-identical.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RNG = 0
SHOW_STATS = True

OUT = Path("figures"); OUT.mkdir(exist_ok=True)
INPUT = Path("../IPSC_DAY60_batch3_expression_all_genes.csv")   # Cuffdiff FPKM matrix
DIFF  = Path("../sample1.vs.sample3/gene_exp.diff")             # Cuffdiff statistics

In [ ]:
# the two day-60 conditions compared in this panel
conditions = {
    "sample1": "5% FBS gfp",
    "sample3": "VEGF+SB+3% FBS",
}

data = pd.read_csv(INPUT, index_col=0)
by_gene = data.set_index("gene_official")
by_gene = by_gene[~by_gene.index.duplicated(keep="first")]

sample_cols = [f"{s}-{r}" for s in conditions for r in range(3)]
expr = by_gene[sample_cols]
groups = pd.Series([conditions[c.split("-")[0]] for c in sample_cols],
                   index=sample_cols, name="group")
print(expr.shape, "genes x", len(sample_cols), "samples")
groups

In [ ]:
cell_markers = {
    "Endothelial Cells": ["FABP4", "GPIHBP1", "CD34", "PECAM1", "VWF", "VEGFA"],
    "Fibroblasts": ["SERPINF1", "RGS5", "MMP7", "LUM", "BGN", "PDGFRA"],
}
filtered_cell_markers = {ct: [g for g in ms if g in expr.index]
                         for ct, ms in cell_markers.items()}
genes = sum(filtered_cell_markers.values(), [])

panel = expr.loc[genes]
log_fpkm = np.log10(panel + 1)

df = log_fpkm.T
df["group"] = groups
order = list(conditions.values())
PALETTE = {order[0]: "#8d8dc6", order[1]: "#7cc4a4"}   # periwinkle / mint
df.round(3)

In [ ]:
# Cuffdiff statistics for this comparison
cuff = pd.read_csv(DIFF, sep="\t").set_index("gene")
cuff = cuff[~cuff.index.duplicated(keep="first")]
stats_tbl = cuff.loc[genes, ["value_1", "value_2", "log2(fold_change)",
                             "p_value", "q_value", "status"]]
stats_tbl

In [ ]:
def stars(p):
    if p < 1e-4: return "****"
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 0.05: return "*"
    return "ns"


np.random.seed(RNG)   # sns.stripplot jitters the replicate dots with the global RNG

nrow, ncol = 2, 6
fig, axes = plt.subplots(nrow, ncol, figsize=(17, 8))
rows = []

for ax, gene in zip(axes.flat, genes):
    sns.violinplot(data=df, x="group", y=gene, order=order, hue="group",
                   hue_order=order, palette=PALETTE, legend=False,
                   cut=0, inner=None, linewidth=0.8, ax=ax)
    sns.boxplot(data=df, x="group", y=gene, order=order, width=0.13,
                showcaps=False, showfliers=False, boxprops={"facecolor": "black"},
                medianprops={"color": "white", "linewidth": 1.2},
                whiskerprops={"color": "black", "linewidth": 1.2}, ax=ax)
    sns.stripplot(data=df, x="group", y=gene, order=order, color="black",
                  size=5, jitter=0.12, alpha=0.9, ax=ax)

    p = float(stats_tbl.loc[gene, "p_value"])
    rows.append({"gene": gene,
                 f"FPKM_{order[0]}": stats_tbl.loc[gene, "value_1"],
                 f"FPKM_{order[1]}": stats_tbl.loc[gene, "value_2"],
                 "log2FC": stats_tbl.loc[gene, "log2(fold_change)"],
                 "cuffdiff_p": p, "cuffdiff_q": stats_tbl.loc[gene, "q_value"],
                 "significance": stars(p)})

    lo, hi = df[gene].min(), df[gene].max()
    span = (hi - lo) or 1.0
    if SHOW_STATS:
        y, h = hi + 0.14 * span, 0.05 * span
        ax.plot([0, 0, 1, 1], [y, y + h, y + h, y], lw=1.1, c="0.35")
        ax.text(0.5, y + h * 1.15, stars(p), ha="center", va="bottom",
                fontsize=12, fontweight="bold", color="0.15")
        ax.text(0.5, y + h * 0.05, f"p={p:.3g}", ha="center", va="top",
                fontsize=8, color="0.45")
        ax.set_ylim(lo - 0.08 * span, y + h * 5.5)
    else:
        ax.set_ylim(lo - 0.10 * span, hi + 0.10 * span)

    ax.set_title(gene, fontstyle="italic", fontsize=13, pad=8)
    ax.set_xlabel(""); ax.set_ylabel("")
    sns.despine(ax=ax)

for r in range(nrow):
    axes[r, 0].set_ylabel("log10(FPKM + 1)")
    for c in range(ncol):
        if r < nrow - 1:
            axes[r, c].set_xticklabels([])
        else:
            axes[r, c].set_xticklabels(order, rotation=20, ha="right")

for r, label in enumerate(filtered_cell_markers):
    box = axes[r, 0].get_position()
    fig.text(0.005, (box.y0 + box.y1) / 2, label, rotation=90,
             va="center", ha="left", fontsize=12, fontweight="bold", color="0.25")

fig.suptitle("Endothelial vs fibroblast markers — iPSC-EC media conditions",
             fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0.025, 0, 1, 0.96])

stem = "iPSC_EC_Day60_marker_violin"
fig.savefig(OUT / f"{stem}.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT / f"{stem}.pdf", bbox_inches="tight")
pd.DataFrame(rows).to_csv(OUT / f"{stem}_stats.csv", index=False)
pd.DataFrame(rows)